This script calculates pairwise distances between alphas and betas in 3 different ways. 

Needs to be run in the python<3.11 environment (environment_distances.yml)

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
# from scipy.stats import pearsonr
# from scipy.spatial import distance
# from scipy.cluster import hierarchy

In [2]:
import functions.pwdistances as pw

For TCRdist, we use the following: `pwseqdist`

this is the function in the background of tcrdist3
my understanding is that tcrdist3 iterates over all these components and calculates a dist score for each
then it combines all the dist scores with some weighting
https://github.com/kmayerb/tcrdist3/blob/55d906b19e4c5038f5fdde843eb2edf8293efd88/tcrdist/repertoire.py#L312
https://github.com/kmayerb/tcrdist3/blob/55d906b19e4c5038f5fdde843eb2edf8293efd88/tcrdist/rep_funcs.py#L33
the function in the tcrdist repo is a wrapper to this: 
https://github.com/agartland/pwseqdist/blob/852e159ad12582973bfbf23dd33fef068723742e/pwseqdist/nb_metrics.py#L201
which is (I think) essentially an implementation of the original tcrdist function

In [ ]:
def calculate_dists_and_plot(epdf):
    dijA_triplet = pw.triplet_diversity(epdf['cdr3a_IMGTgaps'].str.replace('-','').tolist()).round(2)*100
    dijB_triplet = pw.triplet_diversity(epdf['cdr3b_IMGTgaps'].str.replace('-','').tolist()).round(2)*100
    
    dijA_lev = pw.levenshtein_dist(epdf['cdr3a_IMGTgaps'].str.replace('-','').tolist()).round(0)
    dijB_lev = pw.levenshtein_dist(epdf['cdr3b_IMGTgaps'].str.replace('-','').tolist()).round(0)

    dijA_lev_w = pw.weighted_levenshtein_dist(epdf['cdr3a_IMGTgaps'].str.replace('-','').tolist())
    dijB_lev_w = pw.weighted_levenshtein_dist(epdf['cdr3b_IMGTgaps'].str.replace('-','').tolist())

    dijA_tcrdist = pw.tcrdist_cdr3s(epdf['cdr3a_IMGTgaps'].str.replace('-','').tolist())
    dijB_tcrdist = pw.tcrdist_cdr3s(epdf['cdr3b_IMGTgaps'].str.replace('-','').tolist())
    
    # save everything - I need to do this because pw and julia are not compatible with each other
    # (this is because of numba, and it's a well-known issue)

    dijA_lev1 = pd.DataFrame(dijA_lev)
    dijB_lev1 = pd.DataFrame(dijB_lev)
    dijA_lev1.index = epdf['cdr3a_IMGTgaps'].str.replace('-','').tolist()
    dijA_lev1.columns = epdf['cdr3a_IMGTgaps'].str.replace('-','').tolist()
    dijB_lev1.index = epdf['cdr3b_IMGTgaps'].str.replace('-','').tolist()
    dijB_lev1.columns = epdf['cdr3b_IMGTgaps'].str.replace('-','').tolist()
    dijA_lev1.to_csv('data/output/pairwise_distances/cdr3/dijA_lev_' + epdf.Epitope.unique()[0] + '.csv')
    dijB_lev1.to_csv('data/output/pairwise_distances/cdr3/dijB_lev_' + epdf.Epitope.unique()[0] + '.csv')

    dijA_lev_w1 = pd.DataFrame(dijA_lev_w)
    dijB_lev_w1 = pd.DataFrame(dijB_lev_w)
    dijA_lev_w1.index = epdf['cdr3a_IMGTgaps'].str.replace('-','').tolist()
    dijA_lev_w1.columns = epdf['cdr3a_IMGTgaps'].str.replace('-','').tolist()
    dijB_lev_w1.index = epdf['cdr3b_IMGTgaps'].str.replace('-','').tolist()
    dijB_lev_w1.columns = epdf['cdr3b_IMGTgaps'].str.replace('-','').tolist()
    dijA_lev_w1.to_csv('data/output/pairwise_distances/cdr3/dijA_weighted_lev_' + epdf.Epitope.unique()[0] + '.csv')
    dijB_lev_w1.to_csv('data/output/pairwise_distances/cdr3/dijB_weighted_lev_' + epdf.Epitope.unique()[0] + '.csv')

    dijA_triplet1 = pd.DataFrame(dijA_triplet)
    dijB_triplet1 = pd.DataFrame(dijB_triplet)
    dijA_triplet1.index = epdf['cdr3a_IMGTgaps'].str.replace('-','').tolist()
    dijA_triplet1.columns = epdf['cdr3a_IMGTgaps'].str.replace('-','').tolist()
    dijB_triplet1.index = epdf['cdr3b_IMGTgaps'].str.replace('-','').tolist()
    dijB_triplet1.columns = epdf['cdr3b_IMGTgaps'].str.replace('-','').tolist()
    dijA_triplet1.to_csv('data/output/pairwise_distances/cdr3/dijA_triplet_' + epdf.Epitope.unique()[0] + '.csv')
    dijB_triplet1.to_csv('data/output/pairwise_distances/cdr3/dijB_triplet_' + epdf.Epitope.unique()[0] + '.csv')

    dijA_tcrdist1 = pd.DataFrame(dijA_tcrdist)
    dijB_tcrdist1 = pd.DataFrame(dijB_tcrdist)
    dijA_tcrdist1.index = epdf['cdr3a_IMGTgaps'].str.replace('-','').tolist()
    dijA_tcrdist1.columns = epdf['cdr3a_IMGTgaps'].str.replace('-','').tolist()
    dijB_tcrdist1.index = epdf['cdr3b_IMGTgaps'].str.replace('-','').tolist()
    dijB_tcrdist1.columns = epdf['cdr3b_IMGTgaps'].str.replace('-','').tolist()
    dijA_tcrdist1.to_csv('data/output/pairwise_distances/cdr3/dijA_tcrdist_' + epdf.Epitope.unique()[0] + '.csv')
    dijB_tcrdist1.to_csv('data/output/pairwise_distances/cdr3/dijB_tcrdist_' + epdf.Epitope.unique()[0] + '.csv')

In [4]:
vdj = pd.read_csv('data/vdj_cleaned_subset_for_MI_no-small-study.csv', index_col = 0)
vdj = vdj.loc[vdj['Epitope'] != 'KLGGALQAK'] # because too big - takes forever
print(vdj.head())

for ep in vdj.Epitope.unique():
    print(ep)
    epdf = vdj.loc[vdj['Epitope'] == ep].reset_index()
    print(epdf.shape)
    calculate_dists_and_plot(epdf)

   Unnamed: 0  complex.id Gene-a           CDR3-a       V-a     J-a  \
0          13          14    TRA    CAYTVLGNEKLTF  TRAV38-1  TRAJ48   
1          14          15    TRA  CAVAGYGGSQGNLIF  TRAV12-2  TRAJ42   
2          15          16    TRA     CAVSFGNEKLTF  TRAV12-2  TRAJ48   
3          16          17    TRA  CAVTHYGGSQGNLIF  TRAV12-2  TRAJ42   
4          17          18    TRA    CAGGGGGADGLTF  TRAV12-2  TRAJ45   

       Species     MHC A MHC B MHC class  ... cdr2a_IMGTgaps cdr2b_IMGTgaps  \
0  HomoSapiens  HLA-A*02   B2M      MHCI  ...     QEAY--KQQN     SYD----VKM   
1  HomoSapiens  HLA-A*02   B2M      MHCI  ...     IYS----NGD     SYD----VKM   
2  HomoSapiens  HLA-A*02   B2M      MHCI  ...     IYS----NGD     SYD----VKM   
3  HomoSapiens  HLA-A*02   B2M      MHCI  ...     IYS----NGD     SYD----VKM   
4  HomoSapiens  HLA-A*02   B2M      MHCI  ...     IYS----NGD     SYD----VKM   

    cdr3a_IMGTgaps   cdr3b_IMGTgaps len_cdr3a len_cdr3b len_cdr3a_IMGTgaps  \
0  CAYTVLG--NEKLTF  